# 1.Data Curation & Preproccessing

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/', force_remount=True)

Installing Dependencies and Libraries

In [ ]:
!pip install chembl_webresource_client

import pandas as pd
from chembl_webresource_client.new_client import new_client

**Target Identification**

In [ ]:
target = new_client.target
target_query = target.search("JAK2")
targets = pd.DataFrame.from_dict(target_query)
targets.head()

Reterive Bioactivity data for selected target

In [ ]:
selected_target = targets.target_chembl_id[0]
selected_target

Now retrieve only bioactivity data for JAK2

In [ ]:
activity = new_client.activity
results = activity.filter(target_chembl_id=selected_target).filter(standard_type="IC50")


In [ ]:
df1 = pd.DataFrame.from_dict(results)
df1.head(5)


In [ ]:
df1.standard_type.unique()

In [ ]:
df1.to_csv('bioactivity_raw_data.csv', index=False)

Save the resulting bioactivity data to a CSV file *bioactivityJAK2_raw_data*.csv.

In [ ]:
df1.to_csv('bioactivity_raw_data.csv', index=False)
! cp bioactivity_raw_data.csv "/content/gdrive/My Drive/Colab Notebooks/data"

! ls -l "/content/gdrive/My Drive/Colab Notebooks/data"

! head bioactivity_raw_data.csv

# Bioactivity Data Retrieval (IC50)

Inspect Missing Values

In [ ]:
df1["standard_type"].isna().sum()

Filter Rows with Valid Bioactivity Values

In [ ]:
df2 = df1[df1["standard_value"].notna()]
df2.head()

Assign Classes


In [ ]:
bioactivity_class = []
for value in df2.standard_value:
    value = float(value)
    if value >= 10000:
        bioactivity_class.append("inactive")
    elif value <= 1000:
        bioactivity_class.append("active")
    else:
        bioactivity_class.append("intermediate")


Extract Relevant Columns

In [ ]:
molecule_ids = df2.molecule_chembl_id.tolist()
canonical_smiles = df2.canonical_smiles.tolist()
standard_values = df2.standard_value.tolist()


In [ ]:
data = list(zip(
    molecule_ids,
    canonical_smiles,
    standard_values,
        bioactivity_class,
))



Preprocessed Data Creation



In [ ]:
df3 = pd.DataFrame(
    data,
    columns=[
        "molecule_chembl_id",
        "canonical_smiles",
        "standard_value",
        "bioactivity_class",
    ]
)
df3.head()

Remove Compounds without Valid SMILES. Drop rows with NaN, empty or None SMILES values.

In [ ]:
df3 = df3.dropna(subset=["canonical_smiles"])
df3 = df3[df3["canonical_smiles"].str.lower() != "none"]
df3 = df3[df3["canonical_smiles"].str.strip() != ""]
df3.head()

Save Preprocessed Data

In [ ]:
df3.to_csv("bioactivityJAK2_preprocessed_data.csv", index=False)
!cp bioactivity_preprocessed_data.csv "/content/gdrive/My Drive/Colab Notebooks/data"
!ls "/content/gdrive/My Drive/Colab Notebooks/data"

# 2. Exploratory Data Analysis
Import libraries for EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='ticks')

Bioactivity Dataset (**Preprocessed**)

In [ ]:

df4 = pd.read_csv("bioactivityJAK2_preprocessed_data.csv")
df4.head(10)

Remove N/A Values

In [ ]:
print("Original shape:", df4.shape)

df4 = df4.dropna(subset=[
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_value"
])

# convert IC50 to numeric
df4["standard_value"] = pd.to_numeric(df4["standard_value"], errors="coerce")

df4 = df4.dropna(subset=["standard_value"])

df4 = df4[df4["bioactivity_class"] != 'intermediate']

print("After cleaning:", df4.shape)


In [ ]:
df4.head()

Aggregate Duplicates IC50 (median IC50 per canonical smile)

In [ ]:
df_clean = (
    df4
    .groupby("canonical_smiles", as_index=False)
    .agg({
        "molecule_chembl_id": "first",
        "standard_value": "median",
        "bioactivity_class": "first"
    })
)

print("Before aggregation:", df4.shape[0])
print("After aggregation:", df_clean.shape[0])

df_clean.head()

In [ ]:
df_clean.standard_value.describe()

Convert IC50 to pIC50
Convert IC50 to the negative logarithmic scale which is essentially -log10(IC50). This conversion allows IC50 data to be more uniformly distributed.

In [ ]:
df_clean["pIC50"] = -np.log10(df_clean["standard_value"] * 1e-9)

df_clean.head()


Reassign Activity Labels Based on PIC50
Based on pIC50
Active >= 6
Inactive < 6

In [ ]:
threshold = 6

df_clean["bioactivity_class"] = np.where(
    df_clean["pIC50"] >= threshold,
    "active",
    "inactive"
)

df_clean.head()


Check **Duplicates**

In [ ]:
print("Duplicate SMILES remaining:",
      df_clean["canonical_smiles"].duplicated().sum())

In [ ]:
df_clean.standard_value.describe()

In [ ]:
df_clean.pIC50.describe()

In [ ]:
df_finite_pic50 = df_clean[np.isfinite(df_clean['pIC50'])]

plt.hist(df_finite_pic50["pIC50"], bins=30)
plt.xlabel("pIC50")
plt.ylabel("Count")
plt.title("pIC50 Distribution")
plt.show()
plt.savefig('histogram_pic50.pdf')

In [ ]:
plt.figure(figsize=(5.5, 5.5))

sns.countplot(x="bioactivity_class", data= df_clean, hue="bioactivity_class", edgecolor='black')

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('Frequency', fontsize=14, fontweight='bold')

## Lipinski's Descriptor

The Lipinski's Rule stated the following:

Molecular weight < 500 Dalton
Octanol-water partition coefficient (LogP) < 5
Hydrogen bond donors < 5
Hydrogen bond acceptors < 10

In [ ]:
!pip install rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from rdkit.Chem import rdMolDescriptors

In [ ]:
df_no_smiles = df_clean.drop(columns='canonical_smiles')

smiles = []

for i in df_clean.canonical_smiles.tolist():
  cpd = str(i).split('.')
  cpd_longest = max(cpd, key=len)
  smiles.append(cpd_longest)

smiles = pd.Series(smiles, name='canonical_smiles')

df_clean_smiles = pd.concat([df_no_smiles, smiles], axis=1)

df_clean_smiles

Calculate descriptors

In [ ]:
def lipinski(smiles, verbose=False):

    moldata= []
    for elem in smiles:
        mol=Chem.MolFromSmiles(elem)
        moldata.append(mol)

    baseData= np.arange(1,1)
    i=0
    for mol in moldata:

        desc_MolWt = Descriptors.MolWt(mol)
        desc_MolLogP = Descriptors.MolLogP(mol)
        desc_NumHDonors = Lipinski.NumHDonors(mol)
        desc_NumHAcceptors = Lipinski.NumHAcceptors(mol)

        row = np.array([desc_MolWt,
                        desc_MolLogP,
                        desc_NumHDonors,
                        desc_NumHAcceptors])

        if(i==0):
            baseData=row
        else:
            baseData=np.vstack([baseData, row])
        i=i+1

    columnNames=["MW","LogP","NumHDonors","NumHAcceptors"]
    descriptors = pd.DataFrame(data=baseData,columns=columnNames)

    return descriptors

In [ ]:
df_lipinski = lipinski(df_clean_smiles.canonical_smiles)
df_lipinski.head()

In [ ]:
df_lipinski.shape

## **Combine Both datasets**

In [ ]:
df_clean_smiles.head()

In [ ]:
df_lipinski.head()

In [ ]:
df_combined = pd.concat([df_clean_smiles, df_lipinski], axis=1)
df_combined.head()

In [ ]:
df_combined = df_combined.drop(columns="standard_value")
df_combined.head()


In [ ]:
# Save CSV
df_combined.to_csv("df_lipinski.csv", index=False)

### Chemical Space Analysis For Lipinski Descriptors

Barplot of the bioactivity classes

In [ ]:
plt.figure(figsize=(5.5, 5.5))

sns.countplot(x="bioactivity_class", data= df_combined, hue="bioactivity_class", edgecolor='black')

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('Frequency', fontsize=14, fontweight='bold')

plt.savefig('barplot_bioactivity_class.pdf')

Boxplot of the bioactivity classes for PIC50

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(5.5, 5.5))

sns.boxplot(x = "bioactivity_class", y = "pIC50", data = df_combined, hue="bioactivity_class")

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('pIC50 value', fontsize=14, fontweight='bold')

Scatter of Molecular Weight vs Solubility (LogP)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df_combined_finite = df_combined[np.isfinite(df_combined['pIC50'])]

plt.figure(figsize=(5.5, 5.5))

sns.scatterplot(x='MW', y='LogP', data=df_combined_finite, hue='bioactivity_class', size='pIC50', edgecolor='black', alpha=0.7)

plt.xlabel('MW', fontsize=14, fontweight='bold')
plt.ylabel('LogP', fontsize=14, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0)
plt.savefig('scatter_plot_MW_vs_LogP.pdf')

Statistical analysis (Mann-Whitney U Test)

In [ ]:
def mannwhitney(descriptor, df_combined, verbose=False):
    """
    Perform Mann-Whitney U test between active and inactive compounds
    for a given descriptor.

    Parameters:
    - descriptor : str, column name of the descriptor
    - df_combined : pandas DataFrame, must have columns [descriptor, bioactivity_class]
    - verbose : bool, if True prints the test statistics

    Returns:
    - results : pandas DataFrame with test statistics, p-value, and interpretation
    """
    from numpy.random import seed
    from scipy.stats import mannwhitneyu
    import pandas as pd

    # set seed for reproducibility
    seed(1)

    # select only relevant columns
    df = df_combined[[descriptor, 'bioactivity_class']]

    # separate active and inactive compounds
    active = df[df['bioactivity_class'] == 'active'][descriptor]
    inactive = df[df['bioactivity_class'] == 'inactive'][descriptor]

    # perform Mann-Whitney U test
    stat, p = mannwhitneyu(active, inactive)

    if verbose:
        print(f"Descriptor: {descriptor}")
        print(f"Statistics={stat:.3f}, p={p:.3f}")

    # interpret result
    alpha = 0.05
    if p > alpha:
        interpretation = 'Same distribution (fail to reject H0)'
    else:
        interpretation = 'Different distribution (reject H0)'

    # store results in a DataFrame
    results = pd.DataFrame({
        'Descriptor': descriptor,
        'Statistics': stat,
        'p': p,
        'alpha': alpha,
        'Interpretation': interpretation
    }, index=[0])

    # save results to CSV
    filename = 'mannwhitneyu_' + descriptor + '.csv'
    results.to_csv(filename, index=False)

    return results

pIC50

In [ ]:
mannwhitney("pIC50", df_combined, verbose=True)

Molecular Weight

In [ ]:
mannwhitney("MW", df_combined, verbose=True)

Solubility LogP

In [ ]:
mannwhitney("LogP", df_combined, verbose=True)



Number of Hydrogen Donors



In [ ]:
mannwhitney("NumHDonors", df_combined, verbose=True)

Number of Hydrogen Acceptors

In [ ]:
mannwhitney("NumHAcceptors", df_combined, verbose=True)

### **Combine All Statistical Results**

In [ ]:
import pandas as pd
import glob
import os

# Get list of all Mann-Whitney CSV files in current folder
mw_files = glob.glob("mannwhitneyu_*.csv")

# Combine them into one DataFrame
mw_combined = pd.concat([pd.read_csv(f) for f in mw_files], ignore_index=True)

# Save combined CSV
combined_filename = "mannwhitney_summary.csv"
mw_combined.to_csv(combined_filename, index=False)

print(f"Combined Mann-Whitney CSV saved as {combined_filename}")

In [ ]:
mw_combined

Molecular Weight

In [ ]:
plt.figure(figsize=(5.5, 5.5))

sns.boxplot(x = 'bioactivity_class', y = 'MW', data = df_combined, hue="bioactivity_class")

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('MW', fontsize=14, fontweight='bold')

plt.savefig('boxplot_MW.pdf')

logP

In [ ]:
plt.figure(figsize=(5.5, 5.5))

sns.boxplot(x = "bioactivity_class", y = 'LogP', data = df_combined, hue="bioactivity_class")

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('LogP', fontsize=14, fontweight='bold')

plt.savefig('boxplot_LogP.pdf')

NumHDonors

In [ ]:
plt.figure(figsize=(5.5, 5.5))

sns.boxplot(x = "bioactivity_class", y = "NumHDonors", data = df_combined, hue="bioactivity_class")

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('NumHDonors', fontsize=14, fontweight='bold')

plt.savefig('boxplot_NumHDonors.pdf')

NumHAcceptors

In [ ]:
plt.figure(figsize=(5.5, 5.5))

sns.boxplot(x = "bioactivity_class", y = "NumHAcceptors", data = df_combined, hue="bioactivity_class")

plt.xlabel('Bioactivity class', fontsize=14, fontweight='bold')
plt.ylabel('NumHAcceptors', fontsize=14, fontweight='bold')

plt.savefig('boxplot_NumHAcceptors.pdf')


Save & Downlaod Results

In [ ]:
! zip -r EDA_results.zip . -i *df_lipinski.csv *mannwhitney_summary.csv *.pdf

# 3. Descriptor Calculation

PaDELPy is a Python wrapper for the PaDEL-Descriptor (molecular descriptor calculation) software.

It provide the following descriptors/fingerprint:

1444 - 2D Descriptors
431 - 3D Descriptors
881 bits - PubChem Fingerprints

### Installing Dependencies & libraries

In [ ]:
!pip install padelpy

import pandas as pd
import numpy as np
from google.colab import files
from padelpy import padeldescriptor

df = pd.read_csv('df_lipinski.csv')
df.head()




In [ ]:
data = df[['canonical_smiles', 'molecule_chembl_id']]
data.head()

Convert to .smi format

In [ ]:
df_smi = data['canonical_smiles'].to_csv('smiles_chembl.smi', index=None, header=None)

In [ ]:
! cat smiles_chembl.smi | head

Calculate molecular Pubchem Fingerprints using "padeldescriptor" function

In [ ]:
padeldescriptor(mol_dir= "smiles_chembl.smi",
                d_file='pubchem_fingerprints.csv',
                fingerprints = True,
                retainorder= True,
                #removesalt = True, standardizetautomers = True, standardizenitro=True
                )

In [ ]:
!ls -lh pubchem_fingerprints.csv

In [ ]:
df_fingerprint = pd.read_csv("pubchem_fingerprints.csv")
df_fingerprint.head()

## Prepare Dataset for ML

In [ ]:
df.head()

In [ ]:
# Select only the columns we need for ML
meta_cols = df[['molecule_chembl_id', 'bioactivity_class', 'pIC50']]

# Reset index to ensure proper alignment
meta_cols = meta_cols.reset_index(drop=True)
df_fingerprint = df_fingerprint.reset_index(drop=True)

# Combine meta data with fingerprints
combined_df = pd.concat([meta_cols, df_fingerprint.drop(df_fingerprint.columns[0], axis=1)], axis=1)

# Inspect the first few rows
combined_df.head()


Save and download the dataset

In [ ]:
# Save as CSV
combined_df.to_csv("QSAR_dataset.csv", index=False)
print("Combined dataset saved as QSAR_dataset.csv")

# Download file in Colab
files.download("QSAR_dataset.csv")

# Calculate other fingerprints

Download xml Files from Github

In [ ]:
!wget https://github.com/AI-Biotechnology-Bioinformatics/Drug_Discovery_AI_Course_2026/raw/main/padel_descriptors_xml.zip

!unzip padel_descriptors_xml.zip


Calculate Fingerprints

In [ ]:
# Specify the XML file for SubstructureFingerprinter directly
Substruc_fp = "SubstructureFingerprinter.xml"

# Calculate Substructure fingerprints, (extra-part)
padeldescriptor(
    mol_dir='smiles_chembl.smi',
    d_file='Substructure_fingerprints.csv',
    fingerprints=True,
    descriptortypes= Substruc_fp,
    retainorder=True
    # removesalt=True, standardizetautomers=True
)

#4. QSAR Modeling

Random forest Regression model to predict pIC50 values


In [ ]:
# Install LazyPredict
!pip install lazypredict
# Install SHAP
!pip install shap

In [ ]:
# Import libraries
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.utils import shuffle
from lazypredict.Supervised import LazyRegressor
import shap

###Loading the Dataset

In [ ]:
df = pd.read_csv('QSAR_dataset.csv')
df.head()

Dataset overview:
molecule_chembl_id: Unique molecule ID
bioactivity_class: Active/inactive class
pIC50: Continuous potency value (4-9)
PubchemFP0 to PubchemFP880: 881 binary fingerprints

Features (X) and Target (y)


In [ ]:
# Exclude non-feature columns
non_feature_cols = ['molecule_chembl_id', 'bioactivity_class', 'pIC50']
X = df.drop(columns=non_feature_cols)
print(X.shape)

In [ ]:
# Target variable
y_reg = df['pIC50']
y_reg


Feature Selection – Variance Threshold
Not all 881 fingerprints are informative. Low-variance features (mostly 0 or 1) add noise. So we remove them.

In [ ]:
# Apply variance threshold
selection = VarianceThreshold(threshold=(0.8*(1-0.8)))  # Threshold = 0.16
X_var = selection.fit_transform(X)

# Extract the correct feature names
selected_mask = selection.get_support()
feature_names = X.columns[selected_mask]

# Convert immediately to DataFrame
X_var = pd.DataFrame(X_var, columns=feature_names)

print('After variance threshold:', X_var.shape)

### Split Data into Training and Test Set

In [ ]:
# Split into 80/20 train/test
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_var, y_reg, test_size=0.2, random_state=123
)
print(f'Training set size: {X_train_reg.shape[0]} molecules')
print(f'Testing set size: {X_test_reg.shape[0]} molecules')

Build and Train Random Forest Model

In [ ]:
np.random.seed(123)

rf_model = RandomForestRegressor(n_estimators=200)

# Filter out infinite values from y_train_reg and corresponding X_train_reg
finite_indices = np.isfinite(y_train_reg)
X_train_reg_filtered = X_train_reg[finite_indices]
y_train_reg_filtered = y_train_reg[finite_indices]

# Train model using filtered data
rf_model.fit(X_train_reg_filtered, y_train_reg_filtered)

Evaluate Model's Performance on Test Set

In [ ]:
finite_indices_test = np.isfinite(y_test_reg)
X_test_reg_filtered = X_test_reg[finite_indices_test]
y_test_reg_filtered = y_test_reg[finite_indices_test]

r2 = rf_model.score(X_test_reg_filtered, y_test_reg_filtered)
print(f'Random Forest R² score: {r2:.4f}')

**R² interpretation:**

R² < 0.4 → Poor fit.

R² < 0.5-0.7 → Moderate/Acceptable.

R² > 0.9 → Very good/High fit

Make Predictions on test set

In [ ]:
# Predict on test set
y_pred_reg = rf_model.predict(X_test_reg)

In [ ]:
y_pred_reg

*Visualize Predictions vs Actual Values*

In [ ]:
ax = sns.regplot(x=y_test_reg, y=y_pred_reg, scatter_kws={'alpha':0.4})
ax.set_xlabel('Experimental pIC50', fontsize='large', fontweight='bold')
ax.set_ylabel('Predicted pIC50', fontsize='large', fontweight='bold')
plt.show()

Y-Randomization – Validate Model

In [ ]:
n_iterations = 5
random_r2_scores = []

# Filter y_train_reg to remove infinite values for Y-randomization
finite_indices_y_train = np.isfinite(y_train_reg)
X_train_reg_finite = X_train_reg[finite_indices_y_train]
y_train_reg_finite = y_train_reg[finite_indices_y_train]

# Filter y_test_reg and X_test_reg as well, in case they contain infinites
finite_indices_y_test = np.isfinite(y_test_reg)
X_test_reg_finite = X_test_reg[finite_indices_y_test]
y_test_reg_finite = y_test_reg[finite_indices_y_test]

for i in range(n_iterations):
    y_train_shuffled = shuffle(y_train_reg_finite, random_state=i)
    rf_random = RandomForestRegressor(n_estimators=200, random_state=42)
    rf_random.fit(X_train_reg_finite, y_train_shuffled)
    y_pred_random = rf_random.predict(X_test_reg_finite)
    r2_random = r2_score(y_test_reg_finite, y_pred_random)
    random_r2_scores.append(r2_random)

print('\nY-Randomization Test Results:')
print(f'Mean R² with shuffled Y: {np.mean(random_r2_scores):.4f}')
print(f'Actual RF R²: {r2:.4f}')

Interpretation

Actual RF R2 = 0.7607

Validation Random R2 = -0.2139 (~ -0.2, acceptable)

### *LazyPredict – Compare Multiple Regression Models*

In [ ]:
print('\n' + '='*60)
print('LAZYPREDICT - COMPARING MULTIPLE REGRESSION MODELS')
print('='*60)

# Initialize LazyRegressor
reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)

# Fit all models using the filtered data
models, predictions = reg.fit(X_train_reg_filtered, X_test_reg_filtered, y_train_reg_filtered, y_test_reg_filtered)

# Display results
print(models.head(10))  # Top 10 models

### Visualize Top Models

In [ ]:
top_models = models.head(10)

plt.figure(figsize=(10, 6))

# Define distinct colors for each bar
colors = ['red', 'blue', 'green', 'orange', 'purple', 'cyan', 'magenta', 'lime', 'brown', 'pink']

# Vertical bar plot with multiple colors and alpha for transparency
plt.bar(top_models.index, top_models['R-Squared'], color=colors, alpha=0.7)

plt.ylabel('R² Score', fontsize=12, fontweight='bold')
plt.xlabel('Model', fontsize=12, fontweight='bold')
plt.title('Top 10 Models from LazyPredict', fontsize=14, fontweight='bold')

# Rotate x-axis labels for readability
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

Compare Best Model with Random Forest

In [ ]:
best_model_name = models.index[0]
best_r2 = models['R-Squared'].iloc[0]
print(f'Best model from LazyPredict: {best_model_name}')
print(f'Best R² score: {best_r2:.4f}')
print(f'Random Forest R²: {r2:.4f}')
print(f'Improvement: {(best_r2 - r2)*100:.2f}%')

# Check Random Forest rank
if 'RandomForestRegressor' in models.index:
    rf_rank = models.index.get_loc('RandomForestRegressor') + 1
    print(f'Random Forest ranked: {rf_rank} out of {len(models)}')

Random Forest Feature Importances using SHAP values

In [ ]:
# Create an explainer and calculate SHAP values for the test set
explainer = shap.Explainer(rf_model)
shap_values = explainer(X_test_reg)

# Summary plot
shap.summary_plot(shap_values, X_test_reg, feature_names= feature_names)